# Metis — Train from a text dataset

Train a Μῆτις language model the **normal way**: next-token prediction on a
plain-text corpus you provide. No teacher API, no tunnel — just your data
and the GPU.

The trainer runs a fixed number of steps (`--iters`), saves checkpoints to
Drive, and `--resume` continues a stopped run.

---
## Step 1 — Mount Google Drive
Your dataset and checkpoints live here.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

---
## Step 2 — Clone repo + install dependencies

In [ ]:
import os, subprocess

REPO = "https://github.com/iamasrakib/Metis.git"
METIS_DIR = "/content/Metis"

if os.path.isdir(METIS_DIR):
    subprocess.run(['git', '-C', METIS_DIR, 'pull', '--quiet'], check=True)
else:
    subprocess.run(['git', 'clone', '--quiet', REPO, METIS_DIR], check=True)
os.chdir(METIS_DIR)

subprocess.run(['pip', 'install', '-q', 'numpy', 'tqdm', 'tiktoken', 'tokenizers'], check=True)

import torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'
print(f'Done. GPU: {gpu}')

---
## Step 3 — Pick your dataset
Put your own `.txt` corpus in Drive (or upload it to this runtime) and the
cell below will use it. If it can't find one, it falls back to a small
corpus shipped with the repo so the notebook still runs.

In [ ]:
import os
from google.colab import userdata

# ── Your dataset ──────────────────────────────────────────────────────────
# Option A: put a plain-text corpus (.txt) in Drive and set the path below.
# Option B: upload it to this runtime with the 📂 Files sidebar and set the
#           path to the /content/… location.
# Option C: set a Colab secret named DATASET (key icon → + New secret).
DRIVE_DATASET = "/content/drive/MyDrive/Metis/my_dataset.txt"

# Fallback so the notebook runs out of the box if no dataset is found.
REPO_FALLBACK = "data/cow_all.txt"

def _secret(name, default=""):
    try:
        return userdata.get(name)
    except Exception:
        return default

DATASET = _secret("DATASET", "") or DRIVE_DATASET
if not os.path.isfile(DATASET):
    print(f"{DATASET} not found — falling back to repo corpus {REPO_FALLBACK}")
    DATASET = REPO_FALLBACK

print(f"Dataset: {DATASET} ({os.path.getsize(DATASET):,} bytes)")

---
## Step 4 — Start training
Runs `metis train` on the dataset. Ctrl+C stops it (saves first); re-running
the cell resumes from the last checkpoint.

In [ ]:
import os, subprocess

# Link checkpoints to Drive
DRIVE_BASE = "/content/drive/MyDrive/Metis"
CKPT_DIR   = "checkpoints_train"
os.makedirs(DRIVE_BASE, exist_ok=True)
drive_ckpt = os.path.join(DRIVE_BASE, CKPT_DIR)
os.makedirs(drive_ckpt, exist_ok=True)
local_link = os.path.abspath(CKPT_DIR)
if not os.path.lexists(local_link):
    os.symlink(drive_ckpt, local_link)
    print(f"Linked -> {drive_ckpt}")

# Train normally from the dataset. Ctrl+C to stop (it saves first); re-run
# this cell to resume from where it stopped.
# Tune below: --preset tiny|small|medium|large, --iters, --lr, --batch-size.
result = subprocess.run(
    ["python", "-u", "-m", "metis.cli", "train",
     "--checkpoint-dir", "checkpoints_train",
     "--dataset", DATASET,
     "--preset", "tiny",
     "--tokenizer", "cl100k_base",
     "--iters", "5000",
     "--resume"]
)
if result.returncode != 0:
    print(f"\nTraining exited with error code {result.returncode}")
    print("Check the traceback above — the most common cause is a wrong --dataset path.")

---
**Stop:** `Ctrl+C` in the run cell (the checkpoint saves first).
**Resume:** re-run the cell — `--resume` picks up from the last checkpoint.
**Bigger corpus / model:** raise `--iters`, or switch `--preset` to `small`/`medium`.
**Test the model:** copy the `checkpoints_train` folder to your PC and run
`metis chat --checkpoint-dir checkpoints_train` (or `metis generate`).

*This notebook no longer uses distillation — plain `metis train` on your data.*